In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold

# Load the dataset
df = pd.read_csv('phishing_url_features.csv')

In [2]:

pd.set_option('display.max_columns', None)
df

,url,URLLength,Domain,DomainLength,TLD,TLDLength,NoOfSubDomain,NoOfLettersInURL,NoOfDigitsInURL,LetterRatioInURL,DigitRatioInURL,NoOfSpecialCharsInURL,PathLength,IsHTTPS,HasIPAddress,HasHyphenInDomain,HasSuspiciousKeyword,DomainEntropy,DomainHasNumber,label
0,https://blockchaoin.info/#/,27,blockchaoin,11,info,4,0,20,0,0.740741,0.000000,7,1,1,0,0,0,3.095795,0,1
1,https://www.insects.org/ced4/crush_freaks.html,46,insects,7,org,3,0,36,1,0.782609,0.021739,9,23,1,0,0,0,2.521641,0,0
2,https://www.elpasotimes.com/story/opinion/edit...,118,elpasotimes,11,com,3,0,83,16,0.703390,0.135593,19,91,1,0,0,0,3.095795,0,0
3,http://direct-certs.bankofamerica.com.techdbas...,147,techdbaseurl46,14,cn,2,3,71,63,0.482993,0.428571,13,22,0,0,0,1,3.664498,1,1
4,https://gotham-magazine.com/lalique-unveils-ep...,53,gotham-magazine,15,com,3,0,44,0,0.830189,0.000000,9,26,1,0,1,0,3.323231,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5183045,http://myapple-login.com.daflonpneus.com.br/up...,90,daflonpneus,11,br,2,2,57,21,0.633333,0.233333,12,47,0,0,0,1,3.277613,0,1
5183046,https://www.portoseguro.com.br/seguro-imobiliaria,49,portoseguro,11,br,2,0,41,0,0.836735,0.000000,8,19,1,0,0,0,2.845351,0,0
5183047,http://shopping4u.in/js/ceo/Arch/sb/,36,shopping4u,10,in,2,0,26,1,0.722222,0.027778,9,16,0,0,0,0,3.121928,1,1
5183048,http://allfreecounter.com/,26,allfreecounter,14,com,3,0,21,0,0.807692,0.000000,5,1,0,0,0,1,3.182006,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5183050 entries, 0 to 5183049
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   url                    object 
 1   URLLength              int64  
 2   Domain                 object 
 3   DomainLength           int64  
 4   TLD                    object 
 5   TLDLength              int64  
 6   NoOfSubDomain          int64  
 7   NoOfLettersInURL       int64  
 8   NoOfDigitsInURL        int64  
 9   LetterRatioInURL       float64
 10  DigitRatioInURL        float64
 11  NoOfSpecialCharsInURL  int64  
 12  PathLength             int64  
 13  IsHTTPS                int64  
 14  HasIPAddress           int64  
 15  HasHyphenInDomain      int64  
 16  HasSuspiciousKeyword   int64  
 17  DomainEntropy          float64
 18  DomainHasNumber        int64  
 19  label                  int64  
dtypes: float64(3), int64(14), object(3)
memory usage: 790.9+ MB


**DATA SPLITTING**

In [4]:
selected_features_extended = [

    "DomainLength",
    "PathLength",
    "TLDLength",
    "NoOfSubDomain",
    "LetterRatioInURL",
    "NoOfSpecialCharsInURL",
    "IsHTTPS",
    "HasIPAddress",
    "HasHyphenInDomain",
    "NoOfDigitsInURL",
    "HasSuspiciousKeyword",
    "DomainEntropy",
    "DomainHasNumber",
]

X = df[selected_features_extended]

y = df["label"]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nTrain distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

X_train shape: (4146440, 13)
X_test shape: (1036610, 13)
y_train shape: (4146440,)
y_test shape: (1036610,)

Train distribution (%):
label
0    55.6
1    44.4
Name: proportion, dtype: float64

Test distribution (%):
label
0    55.6
1    44.4
Name: proportion, dtype: float64


In [6]:
import numpy as np

num_cols = X_train.select_dtypes(include=['number']).columns

skewness_train = X_train[num_cols].skew()

skewed_features = skewness_train[
    (skewness_train > 0.5) | (skewness_train < -0.5)
].index

print("skewed features:")
print(skewed_features)

skewed features:
Index(['DomainLength', 'PathLength', 'TLDLength', 'NoOfSubDomain',
       'LetterRatioInURL', 'NoOfSpecialCharsInURL', 'HasIPAddress',
       'HasHyphenInDomain', 'NoOfDigitsInURL', 'HasSuspiciousKeyword',
       'DomainEntropy', 'DomainHasNumber'],
      dtype='object')


In [7]:
X_train_log = X_train.copy()
X_test_log = X_test.copy()

for col in skewed_features:
    
    if (X_train_log[col] < 0).any():
        
        shift = abs(X_train_log[col].min()) + 1
        
        X_train_log[col] = np.log1p(X_train_log[col] + shift)
        X_test_log[col] = np.log1p(X_test_log[col] + shift)
    
    else:
        
        X_train_log[col] = np.log1p(X_train_log[col])
        X_test_log[col] = np.log1p(X_test_log[col])

**SCALING**

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = X_train_log.select_dtypes(include=['number']).columns

X_train_scaled = X_train_log.copy()
X_test_scaled = X_test_log.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train_log[num_cols])
    
X_test_scaled[num_cols] = scaler.transform(X_test_log[num_cols])

**FEATURE SELECTION**

In [9]:
# hanya kolom numerik
X_features = X_train_log.select_dtypes(include=['number'])

print(X_features.columns)

Index(['DomainLength', 'PathLength', 'TLDLength', 'NoOfSubDomain',
       'LetterRatioInURL', 'NoOfSpecialCharsInURL', 'IsHTTPS', 'HasIPAddress',
       'HasHyphenInDomain', 'NoOfDigitsInURL', 'HasSuspiciousKeyword',
       'DomainEntropy', 'DomainHasNumber'],
      dtype='object')


In [10]:
spearman_corr_table = X_features.corr(method='spearman')

print("=== Spearman Feature Correlation ===")
display(spearman_corr_table)

=== Spearman Feature Correlation ===


,DomainLength,PathLength,TLDLength,NoOfSubDomain,LetterRatioInURL,NoOfSpecialCharsInURL,IsHTTPS,HasIPAddress,HasHyphenInDomain,NoOfDigitsInURL,HasSuspiciousKeyword,DomainEntropy,DomainHasNumber
DomainLength,1.000000,0.056202,0.171263,-0.139381,0.182730,0.014186,-0.071191,0.133942,0.196128,0.000655,0.085836,0.883070,-0.002969
PathLength,0.056202,1.000000,-0.027661,-0.181923,0.053865,0.646694,0.119781,0.011015,-0.009609,0.259600,0.059471,0.044210,-0.086569
TLDLength,0.171263,-0.027661,1.000000,-0.068667,0.148567,-0.089128,0.089666,-0.178244,-0.027491,-0.094790,-0.021499,0.194193,-0.051244
NoOfSubDomain,-0.139381,-0.181923,-0.068667,1.000000,-0.164692,0.097726,-0.324671,-0.063638,0.011192,0.229681,0.382301,-0.118070,0.202975
LetterRatioInURL,0.182730,0.053865,0.148567,-0.164692,1.000000,-0.176154,0.187620,-0.199657,-0.046379,-0.812554,-0.118652,0.192554,-0.248861
NoOfSpecialCharsInURL,0.014186,0.646694,-0.089128,0.097726,-0.176154,1.000000,0.035952,0.036571,0.073583,0.443734,0.238327,0.004243,0.010935
IsHTTPS,-0.071191,0.119781,0.089666,-0.324671,0.187620,0.035952,1.000000,-0.109538,-0.059303,-0.155819,-0.319588,-0.045955,-0.162891
HasIPAddress,0.133942,0.011015,-0.178244,-0.063638,-0.199657,0.036571,-0.109538,1.000000,-0.042714,0.164798,0.087789,-0.034487,0.436304
HasHyphenInDomain,0.196128,-0.009609,-0.027491,0.011192,-0.046379,0.073583,-0.059303,-0.042714,1.000000,0.017743,0.070275,0.219657,0.007302
NoOfDigitsInURL,0.000655,0.259600,-0.094790,0.229681,-0.812554,0.443734,-0.155819,0.164798,0.017743,1.000000,0.219716,-0.026886,0.238992


In [11]:
pearson_corr_table = X_features.corr(method='pearson')

print("=== Pearson Feature Correlation ===")
display(pearson_corr_table)

=== Pearson Feature Correlation ===


,DomainLength,PathLength,TLDLength,NoOfSubDomain,LetterRatioInURL,NoOfSpecialCharsInURL,IsHTTPS,HasIPAddress,HasHyphenInDomain,NoOfDigitsInURL,HasSuspiciousKeyword,DomainEntropy,DomainHasNumber
DomainLength,1.000000,0.051963,0.172348,-0.088618,0.110725,0.022372,-0.071032,0.113460,0.202576,0.015927,0.091741,0.872398,-0.007152
PathLength,0.051963,1.000000,-0.032321,-0.152196,0.009174,0.560448,0.126927,0.015118,-0.016406,0.260567,0.067067,0.048903,-0.087101
TLDLength,0.172348,-0.032321,1.000000,-0.064662,0.145079,-0.077361,0.085675,-0.172170,-0.024128,-0.090084,-0.018761,0.189800,-0.041755
NoOfSubDomain,-0.088618,-0.152196,-0.064662,1.000000,-0.176151,0.168246,-0.299650,-0.057016,0.013357,0.288467,0.410151,-0.070902,0.178059
LetterRatioInURL,0.110725,0.009174,0.145079,-0.176151,1.000000,-0.183285,0.194363,-0.307617,-0.036815,-0.784533,-0.129150,0.123900,-0.284600
NoOfSpecialCharsInURL,0.022372,0.560448,-0.077361,0.168246,-0.183285,1.000000,0.028002,0.036036,0.068220,0.471224,0.232188,0.017989,0.009614
IsHTTPS,-0.071032,0.126927,0.085675,-0.299650,0.194363,0.028002,1.000000,-0.109538,-0.059303,-0.167861,-0.319588,-0.039864,-0.162891
HasIPAddress,0.113460,0.015118,-0.172170,-0.057016,-0.307617,0.036036,-0.109538,1.000000,-0.042714,0.155502,0.087789,0.004665,0.436304
HasHyphenInDomain,0.202576,-0.016406,-0.024128,0.013357,-0.036815,0.068220,-0.059303,-0.042714,1.000000,0.022066,0.070275,0.193442,0.007302
NoOfDigitsInURL,0.015927,0.260567,-0.090084,0.288467,-0.784533,0.471224,-0.167861,0.155502,0.022066,1.000000,0.246129,-0.000799,0.221355


In [12]:
import numpy as np
import pandas as pd

threshold = 0.8

corr_matrix = X_features.corr(method="spearman")

high_corr = []

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        
        corr_value = corr_matrix.iloc[i, j]
        
        if abs(corr_value) >= threshold:
            
            high_corr.append({
                "feature_1": corr_matrix.columns[i],
                "feature_2": corr_matrix.columns[j],
                "correlation": corr_value
            })

high_corr_df = pd.DataFrame(high_corr).sort_values(
    by="correlation",
    key=abs,
    ascending=False
)

print("Highly correlated features:")
display(high_corr_df)

Highly correlated features:


,feature_1,feature_2,correlation
1,DomainEntropy,DomainLength,0.883070
0,NoOfDigitsInURL,LetterRatioInURL,-0.812554


**MODELLING**

In [13]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [14]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    
    y_prob = model.predict_proba(X_test)[:,1]

    results = {

        "accuracy": accuracy_score(y_test, y_pred),
        
        "precision": precision_score(y_test, y_pred),
        
        "recall": recall_score(y_test, y_pred),
        
        "f1": f1_score(y_test, y_pred),
        
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    return results

In [15]:
logreg_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logreg_results = evaluate_model(
    logreg_model,
    X_train,
    X_test,
    y_train,
    y_test
)

logreg_results

{'accuracy': 0.8766112617088394,
 'precision': 0.8741555727693611,
 'recall': 0.8435103058958503,
 'f1': 0.8585595647510257,
 'roc_auc': 0.9578317025964471}

In [16]:
xgb_model = XGBClassifier(

    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    
    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    eval_metric="logloss"
)

xgb_results = evaluate_model(
    xgb_model,
    X_train,
    X_test,
    y_train,
    y_test
)

xgb_results

{'accuracy': 0.9065376563992243,
 'precision': 0.8849147830803522,
 'recall': 0.9075076810756548,
 'f1': 0.8960688440389017,
 'roc_auc': 0.9737896542138685}

In [17]:
lgb_model = LGBMClassifier(

    n_estimators=200,
    learning_rate=0.05,
    
    num_leaves=31,
    
    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42
)

lgb_results = evaluate_model(
    lgb_model,
    X_train,
    X_test,
    y_train,
    y_test
)

lgb_results

[LightGBM] [Info] Number of positive: 1840888, number of negative: 2305552
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.218696 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1221
[LightGBM] [Info] Number of data points in the train set: 4146440, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443968 -> initscore=-0.225072
[LightGBM] [Info] Start training from score -0.225072


{'accuracy': 0.9080628201541563,
 'precision': 0.8885845779673687,
 'recall': 0.9065929051631604,
 'f1': 0.8974984162766961,
 'roc_auc': 0.974331732141293}

In [18]:
cat_model = CatBoostClassifier(

    iterations=200,
    depth=6,
    learning_rate=0.05,

    random_state=42,
    verbose=0
)

cat_results = evaluate_model(
    cat_model,
    X_train,
    X_test,
    y_train,
    y_test
)

cat_results

{'accuracy': 0.9041847946672326,
 'precision': 0.8814943542061745,
 'recall': 0.9059823302666974,
 'f1': 0.8935706027138034,
 'roc_auc': 0.9722614064628577}

In [19]:
results_df = pd.DataFrame({

    "Logistic Regression": logreg_results,
    "XGBoost": xgb_results,
    "LightGBM": lgb_results,
    "CatBoost": cat_results

}).T

results_df

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.876611,0.874156,0.843510,0.858560,0.957832
XGBoost,0.906538,0.884915,0.907508,0.896069,0.973790
LightGBM,0.908063,0.888585,0.906593,0.897498,0.974332
CatBoost,0.904185,0.881494,0.905982,0.893571,0.972261


In [25]:
import optuna

c:\Users\LOQ\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
def objective_logreg(trial):

    params = {

        "C": trial.suggest_float("C", 0.001, 10, log=True),

        "solver": trial.suggest_categorical(
            "solver",
            ["lbfgs","liblinear"]
        ),

        "max_iter": 1000
    }

    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = LogisticRegression(**params)

        model.fit(X_tr, y_tr)

        preds = model.predict_proba(X_val)[:,1]

        score = roc_auc_score(y_val, preds)

        scores.append(score)

    return np.mean(scores)


study_logreg = optuna.create_study(direction="maximize")

study_logreg.optimize(objective_logreg, n_trials=30)

best_logreg_params = study_logreg.best_params

[I 2026-03-27 21:40:11,513] A new study created in memory with name: no-name-9f00837a-07f3-4a89-89aa-1756b6531125


In [ ]:
def objective_xgb(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 100, 400),

        "max_depth": trial.suggest_int("max_depth", 3, 10),

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),

        "subsample": trial.suggest_float("subsample", 0.6, 1),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1
        ),

        "gamma": trial.suggest_float("gamma", 0, 5),

        "eval_metric": "logloss",

        "random_state": 42
    }

    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = XGBClassifier(**params)

        model.fit(X_tr, y_tr)

        preds = model.predict_proba(X_val)[:,1]

        score = roc_auc_score(y_val, preds)

        scores.append(score)

    return np.mean(scores)


study_xgb = optuna.create_study(direction="maximize")

study_xgb.optimize(objective_xgb, n_trials=30)

best_xgb_params = study_xgb.best_params

In [ ]:
def objective_lgb(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 100, 400),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            100
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1
        ),

        "random_state": 42
    }

    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = LGBMClassifier(**params)

        model.fit(X_tr, y_tr)

        preds = model.predict_proba(X_val)[:,1]

        score = roc_auc_score(y_val, preds)

        scores.append(score)

    return np.mean(scores)


study_lgb = optuna.create_study(direction="maximize")

study_lgb.optimize(objective_lgb, n_trials=30)

best_lgb_params = study_lgb.best_params

In [ ]:
def objective_cat(trial):

    params = {

        "iterations": trial.suggest_int("iterations", 100, 400),

        "depth": trial.suggest_int("depth", 4, 10),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            10
        ),

        "random_state": 42,

        "verbose": 0
    }

    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(X_tr, y_tr)

        preds = model.predict_proba(X_val)[:,1]

        score = roc_auc_score(y_val, preds)

        scores.append(score)

    return np.mean(scores)


study_cat = optuna.create_study(direction="maximize")

study_cat.optimize(objective_cat, n_trials=30)

best_cat_params = study_cat.best_params

In [ ]:
best_models = {

    "LogReg":
    LogisticRegression(
        **best_logreg_params,
        max_iter=1000
    ),

    "XGBoost":
    XGBClassifier(
        **best_xgb_params,
        eval_metric="logloss"
    ),

    "LightGBM":
    LGBMClassifier(
        **best_lgb_params
    ),

    "CatBoost":
    CatBoostClassifier(
        **best_cat_params,
        verbose=0
    )
}

In [ ]:
final_results = {}

for name, model in best_models.items():

    model.fit(X_train, y_train)

    preds = model.predict_proba(X_test)[:,1]

    score = roc_auc_score(y_test, preds)

    final_results[name] = score


pd.Series(final_results).sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ambil importance
logreg_importance = np.abs(best_models["LogReg"].coef_[0])

xgb_importance = best_models["XGBoost"].feature_importances_

lgb_importance = best_models["LightGBM"].feature_importances_

cat_importance = best_models["CatBoost"].feature_importances_


# buat subplot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))


# Logistic Regression
pd.Series(
    logreg_importance,
    index=X_train.columns
).sort_values().plot.barh(ax=axes[0,0])

axes[0,0].set_title("Logistic Regression Importance")


# XGBoost
pd.Series(
    xgb_importance,
    index=X_train.columns
).sort_values().plot.barh(ax=axes[0,1])

axes[0,1].set_title("XGBoost Importance")


# LightGBM
pd.Series(
    lgb_importance,
    index=X_train.columns
).sort_values().plot.barh(ax=axes[1,0])

axes[1,0].set_title("LightGBM Importance")


# CatBoost
pd.Series(
    cat_importance,
    index=X_train.columns
).sort_values().plot.barh(ax=axes[1,1])

axes[1,1].set_title("CatBoost Importance")


plt.tight_layout()

plt.show()